# Reports: choosing what to show, and where

Without a report, `result.show()` draws an overview and a section per metric. A **report** says exactly
which summaries and charts to show and in what layout. It is a tree of blocks saved in `experiment.json`
with the rest of the experiment, so a UI can edit it too:

| block | shows |
|---|---|
| `Overview()` | status, size, revision, and a table of scenarios and what each changed |
| `Text(markdown=...)` | prose (raw HTML is shown as text, not run) |
| `Table(metric=...)` | one metric's summaries, a row per scenario |
| `Chart(plot=...)` | one plot |
| `MetricSection(metric=...)` | a metric's heading, where it measures, its table and charts |
| `Row(blocks=[...])` | blocks side by side |
| `Section(title=..., blocks=[...])` | a titled group |

`result.save_html(path)` writes the same report as one page to share.

In [1]:
from decikit.experiments import *
from example_projects.experiments.loan_pipeline import applications, build

ENTRY = "example_projects.experiments.loan_pipeline:build"
pipeline = build()   # ratio -> rate -> pricing (secured | unsecured) -> approved, payment
loans = applications(200_000)
loans.head()

income,debt,collateral
f64,f64,f64
3829.0,718.0,1344.0
3454.0,938.0,3913.0
4704.0,1956.0,0.0
3797.0,1960.0,16821.0
2939.0,387.0,0.0


In [2]:
RATE = dict(column="rate", range=(0.05, 0.35), bins=30)
exp = (Experiment("premium-report", pipeline, loans, entry=ENTRY)
       .scenario("premium_8pct", params={"loans": {"pricing": {"unsecured": {"premium": 0.08}}}})
       .metric(Divergence(column="approved"),
               Histogram(name="rate_unsecured", node_id="03ed37e0af6b", **RATE),
               Histogram(name="rate_after", nodes="loans/pricing", **RATE),
               Describe(column="payment"),
               BranchFlows(outcome="approved"))
       .plot(Changes(metric="divergence"), Distribution(metric="rate_unsecured", layout="rows"),
             Distribution(metric="rate_after", layout="side_by_side", width=520),
             Sankey(name="flows", metric="branch_flows", scenario="premium_8pct"))
       .report(
           Overview(),
           Text(markdown="A higher **unsecured premium** reprices loans without collateral. Secured loans "
                         "should not move at all; that is the first thing to check."),
           Section(title="Rates", blocks=[Row(blocks=[Chart(plot="rate_unsecured"), Chart(plot="rate_after")])]),
           Section(title="Approvals", blocks=[Row(blocks=[Chart(plot="divergence"), Table(metric="divergence")])]),
           Section(title="Who goes where", blocks=[Chart(plot="flows")]),
           MetricSection(metric="describe"),
           title="Unsecured premium: rates and approvals",
       ))
result = exp.run()
result.show()

Each plot takes options: `Distribution` and `Bars` take a `layout`, and every plot takes `width` and
`height`. The same histogram four ways:

In [3]:
from decikit.experiments import report
layouts = [Distribution(name=layout, metric="rate_after", layout=layout, width=320)
           for layout in ("overlay", "stacked", "side_by_side", "rows")]
summaries = result.summaries["rate_after"]
for plot in layouts:
    Figure.of(plot.render(summaries)).show()

<vega-lite chart>

<vega-lite chart>

<vega-lite chart>

<vega-lite chart>

The same report as a standalone page. Its charts load Vega from the CDN when the page is opened.

In [4]:
import tempfile
from pathlib import Path

page = result.save_html(Path(tempfile.mkdtemp()) / "report.html")
print(page.name, f"{page.stat().st_size:,} bytes")

report.html 38,300 bytes


And the report as it sits in `experiment.json`: blocks named by `type`, the names of metrics and plots
as plain strings. `validate` checks every name exists before the run.

In [5]:
import json
print(json.dumps(exp.def_.model_dump(mode="json")["report"], indent=1)[:800])

{
 "title": "Unsecured premium: rates and approvals",
 "blocks": [
  {
   "type": "overview"
  },
  {
   "type": "text",
   "markdown": "A higher **unsecured premium** reprices loans without collateral. Secured loans should not move at all; that is the first thing to check."
  },
  {
   "type": "section",
   "title": "Rates",
   "blocks": [
    {
     "type": "row",
     "blocks": [
      {
       "type": "chart",
       "plot": "rate_unsecured"
      },
      {
       "type": "chart",
       "plot": "rate_after"
      }
     ]
    }
   ]
  },
  {
   "type": "section",
   "title": "Approvals",
   "blocks": [
    {
     "type": "row",
     "blocks": [
      {
       "type": "chart",
       "plot": "divergence"
      },
      {
       "type": "table",
       "metric": "divergence"
      }
  


In [6]:
from decikit.experiments import validate
broken = Experiment("b", pipeline, loans).metric(Divergence(column="approved")).report(Chart(plot="rates"))
validate(broken.def_, pipeline)

["the report shows plot 'rates', which the experiment doesn't have"]